In [6]:
import os
import argparse
import numpy as np
import pandas as pd
from decimal import *
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
def update_delta(gamma_j, beta):
    return 1/gamma_j * ((gamma_j*(beta - 1) + 1)**(1/(1 - beta)))

def update_eff(n_a, E_o, S_o, delta_e, k_C, n_d):
    # if E_o - n_a* S_o is positive, the exponential value of this expression will explode in calculation.
    # The original form of efficiency formula contains negative exponent and
    # the simplified form contains the positive exponent
    
    if (E_o - n_a*S_o) >= 0:
        return n_d * (n_a - \
                        ((E_o - n_a*S_o)*n_a*np.exp(-(E_o - n_a * S_o) * k_C * delta_e))/ \
                        (E_o - n_a*S_o*np.exp(-(E_o - n_a*S_o) * k_C * delta_e)))
    else:
        return n_d * (n_a - \
                    ((E_o - n_a*S_o) * n_a)/ \
                        (E_o * np.exp((E_o - n_a * S_o) * k_C * delta_e) - n_a * S_o))
    
def simulate_vals(D_init, E_init, P_init, delta_e, n_d, n_dE, k_C, beta, n_cycle=50):
    S_o, gamma, delta, n_a, eff = [], [], [], [], []
    P_o, E_o, D_e = [P_init], [E_init], [D_init]


    for i in range(n_cycle):
        S_o_j = n_d*D_e[i]
        S_o.append(S_o_j)

        gamma_j = S_o_j/P_o[i]
        gamma.append(gamma_j)

        delta_j = update_delta(gamma_j, beta=beta)
        delta.append(delta_j)

        n_a_j = 1/gamma_j - delta_j
        n_a.append(n_a_j)

        E_o_j = (n_dE**(i+1)) * E_o[0]
        E_o.append(E_o_j)

        eff_j = update_eff(n_a=n_a_j, E_o=E_o_j, S_o=S_o_j, delta_e=delta_e, k_C=k_C, n_d=n_d)
        eff.append(eff_j)
        # print('At cycle {}, PCR efficiency is {}'.format(i, eff_j))

        D_e_j = (eff_j + 1)*D_e[i]
        D_e.append(D_e_j)

        P_o_j = P_o[i] - eff_j * S_o_j
        P_o.append(P_o_j)
    
    return eff, n_a, D_e, S_o, E_o, P_o, gamma, delta

In [3]:
# define constants
k_C = Decimal(15)
n_d = Decimal(1)
n_dE = Decimal(0.99)

beta = Decimal(23)

P_init = Decimal(9.e5)
E_init = Decimal(1e5)
delta_e = Decimal(50)


In [4]:
D_inits = []
D_es = []

for i in tqdm(np.linspace(0.000000000000000000000000000000000000000000000000000000000000000001, 100000, 10000)): #np.linspace(-6, 6, 10000)
    try:
        D_e = simulate_vals(D_init=Decimal(i),#Decimal(10.0**i) 
                        E_init=E_init, 
                        P_init=P_init, 
                        delta_e=delta_e,
                        n_d=n_d,
                        n_dE=n_dE,
                        k_C=k_C,
                        beta=beta,
                        n_cycle=49)[2]
    except DivisionByZero:
        print(f'input {i} skipped due to divisionbyzero')

    # plt.plot(D_e, label = 'modelled curve')
    # plt.legend()
    # plt.grid()

    D_inits.append(np.array([i]))
    D_es.append(np.array([float(str(x)) for x in D_e]))
        


curves_dict = {'D Init': D_inits, 'D Values': D_es}
curves_df = pd.DataFrame(curves_dict)
print(curves_df.shape)

(10000, 2)


In [7]:
print(curves_df.head())
# curves_df.to_csv('data/curves_vary_D_init_params.csv', index=False)

                 D Init                                           D Values
0               [1e-66]  [1e-66, 1e-66, 1e-66, 1e-66, 1e-66, 1e-66, 1e-...
1  [10.001000100010002]  [10.001000100010002, 20.000722379635555, 39.99...
2  [20.002000200020003]  [20.002000200020003, 39.99888997013408, 79.977...
3  [30.003000300030003]  [30.003000300030003, 59.99450404823051, 119.94...
4  [40.004000400040006]  [40.004000400040006, 79.98756588971007, 159.89...


In [8]:
import torch
from torch.utils.data import Dataset

class SynthCurveDataset(Dataset):
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def __getitem__(self, index):
        return torch.tensor(self.inputs[index]).unsqueeze(dim=0), torch.tensor(self.outputs[index])
    
    def __len__(self):
        return len(self.inputs)
    


c:\Users\alexa\anaconda3\envs\cs285_v3\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from sklearn.model_selection import train_test_split

data_train, data_val, labels_train, labels_val = train_test_split(D_es, D_inits, test_size=0.2, random_state=42)
train_dataset = SynthCurveDataset(data_train, labels_train)
val_dataset = SynthCurveDataset(data_val, labels_val)

print(len(train_dataset))
print(len(val_dataset))
print(train_dataset[0])
print(val_dataset[0])

8000
2000
(tensor([[ 92549.2549, 139644.7218, 196818.0741, 261420.0407, 330462.2543,
         401133.1238, 471045.6755, 538317.3047, 601561.8891, 659841.1275,
         712599.7057, 759596.8191, 800840.3796, 836527.0292, 866989.3728,
         892650.8865, 913988.4104, 931501.8124, 945690.2188, 957034.1121,
         965982.5480, 972944.7422, 978285.2975, 982322.3799, 985328.2065,
         987531.2638, 989119.7420, 990245.7392, 991029.8551, 991565.8700,
         991925.2669, 992161.4237, 992313.3610, 992408.9822, 992467.7907,
         992503.0986, 992523.7702, 992535.5586, 992542.0989, 992545.6248,
         992547.4694, 992548.4045, 992548.8633, 992549.0809, 992549.1803,
         992549.2242, 992549.2428, 992549.2503, 992549.2532, 992549.2543]],
       dtype=torch.float64), tensor([92549.2549], dtype=torch.float64))
(tensor([[ 62526.2526,  99684.4346, 147883.5128, 205308.6114, 269250.2325,
         336817.0890, 405367.0406, 472699.5367, 537106.8639, 597353.9143,
         652625.6421, 7024

In [11]:
from resnet import EKGResNetModel
from base import fit_model
from torch.utils.data import DataLoader

NUM_EPOCHS = 20
batch_size = 50

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

example_data, example_label = train_dataset[0]

if len(example_data.shape) == 1:
    n_samples = example_data.shape[0]
    n_channels = 1
else:
    n_channels, n_samples = example_data.shape

n_outputs = example_label.shape[0]
regress = [True]*n_outputs

print("cuda available: " + str(torch.cuda.is_available()))

model = EKGResNetModel(n_channels=n_channels, n_samples=n_samples, n_outputs=n_outputs, num_rep_blocks=8, kernel_size=16, regress=[True]) #original num_rep_blocks 32
fit_model(model, train_dataloader, val_dataloader, save_path="output/resnet_lr1e-5", max_epochs=NUM_EPOCHS, learning_rate=1e-5)

cuda available: True

SEQ_LEN:  6 
OUT_FEATURES:  64 

net thinks it will have
  seq_len     : 6
  last active : 384
-------------------
fitting model:  {'save_path': 'output/resnet_lr1e-5', 'max_epochs': 20, 'learning_rate': 1e-05}
Enumerating batches, (epoch 0)


  0%|          | 0/160 [00:00<?, ?it/s]